<div align="right"><a href="L16-Entwurf-nichtrekursiver-Filter-mittels-Fensterung.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/S16-Design-of-non-recursive-filters-using-windowing.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S15 Window Functions](./S15-Window-Functions.ipynb) | [Exercise 16](./E16-Design-of-non-recursive-filters-using-windowing.ipynb) | [Contents / Overview](./00-Python-Notebooks-Overview-EN.ipynb) | Next: [S17 Remez design (Chebyshev approximation)](./S17-Remez-design-Chebyshev-approximation.ipynb)

</div>

---


# Design of Non-Recursive Filters Using Windowing Techniques

The complete filter design by windowing an impulse response of a time-shifted, ideal low-pass filter is implemented in Python using libraries like `scipy.signal.firwin`.

Calling `b = firwin(m+1, Wn)` generates $m+1$ filter coefficients of a low-pass filter of order `m` and cut-off frequency `Wn`.

For a high-pass filter: `b = firwin(m+1, Wn, pass_zero=False)` yields the corresponding high-pass filter with cut-off frequency `Wn`.

For a band-pass filter: `b = firwin(m+1, [Wn1, Wn2], pass_zero=False)` yields a pass-band filter with cut-off frequencies `Wn1` and `Wn2`.

By default, the Hamming window is used. In addition, the last parameter may be a vector with samples of an arbitrary window function: `b = firwin(m+1, Wn, window=window)`. This vector must cover $m+1$ samples.

---

### a)

Design a **low-pass filter** by setting `m = 50` and `Wn = 0.4`. Compare the absolute value of the spectrum for different window functions.

---

### b)

Design a **high-pass filter** of order `m = 33` and cut-off frequency `Wn = 0.4` using a Hamming window. Why does Python/Scipy generate a warning?

---

### c)

Group delay and linearity of a system are closely connected. Calculate the **group delay** of the low-pass filter from exercise a) (Hamming window). Use Python, e.g., `scipy.signal.group_delay`.

In [ ]:
# BEGIN SOLUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import firwin, freqz, group_delay
from scipy.signal.windows import kaiser

# ----------------------------------------------------------
# Task a: Lowpass Filter Design
# ----------------------------------------------------------

N = 50
cutoff = 0.4

# Windows (MATLAB-like)
box   = np.ones(N+1)
hann  = np.hanning(N+1)
hamm  = np.hamming(N+1)
black = np.blackman(N+1)
kais  = kaiser(N+1, beta=8)

# Base lowpass FIR (no window)
b_base = firwin(N+1, cutoff)

# Apply MATLAB-like windows manually
bR  = b_base * box
bH  = b_base * hann
bHM = b_base * hamm
bB  = b_base * black
bK  = b_base * kais

# Frequency responses
W, HR  = freqz(bR, 1, 512)
_, HH  = freqz(bH, 1, 512)
_, HHM = freqz(bHM, 1, 512)
_, HB  = freqz(bB, 1, 512)
_, HK  = freqz(bK, 1, 512)

# ----------------------------------------------------------
# Plot 1: Rectangular, Hann, Hamming
# ----------------------------------------------------------

plt.figure()
plt.plot(W/np.pi, 20*np.log10(np.abs(HR / np.max(HR))))
plt.plot(W/np.pi, 20*np.log10(np.abs(HH / np.max(HH))), '--')
plt.plot(W/np.pi, 20*np.log10(np.abs(HHM / np.max(HHM))), '-.')
plt.xlabel('Omega/pi')
plt.ylabel('|H(exp(j*Omega))/H(1)| in dB')
plt.grid()
plt.title('Lowpass: - Rectangular   -- Hann   -. Hamming')
plt.axis([0, 1, -150, 0])

# ----------------------------------------------------------
# Plot 2: Blackman, Kaiser
# ----------------------------------------------------------

plt.figure()
plt.plot(W/np.pi, 20*np.log10(np.abs(HB / np.max(HB))), '-')
plt.plot(W/np.pi, 20*np.log10(np.abs(HK / np.max(HK))), '--')
plt.xlabel('Omega/pi')
plt.ylabel('|H(exp(j*Omega))/H(1)| in dB')
plt.grid()
plt.title('Lowpass: - Blackman   -- Kaiser β=8')
plt.axis([0, 1, -150, 0])

# ----------------------------------------------------------
# Task b: Highpass Filter
# ----------------------------------------------------------

N_hp = 33
b_hp = firwin(N_hp, cutoff, pass_zero=False)
W, H = freqz(b_hp, 1, 512)

plt.figure()
plt.plot(W/np.pi, 20*np.log10(np.abs(H / np.max(H))))
plt.grid()
plt.xlabel('Omega/pi')
plt.ylabel('|H(exp(j*Omega))/H(-1)| in dB')
plt.title('Highpass')

# ----------------------------------------------------------
# Task c: Group Delay
# ----------------------------------------------------------

b_gd = firwin(50, cutoff)
w, gd = group_delay((b_gd, 1), 256)

plt.figure()
plt.plot(w/np.pi, gd)
plt.grid()
plt.title('Group Delay in Samples')
plt.xlabel('Omega/pi')
plt.axis([0, 1, 24, 26])

plt.show()
# END SOLUTION

## Solution: Design of Non-Recursive Filters Using Windowing Techniques

### a)

In the following figures, the **amplitude responses** of each filter with order $m=50$ are plotted.

<img src="./pngs/ldsv5a02_a1.png" width="250">
<img src="./pngs/ldsv5a02_a2.png" width="250">

The figures show the normalized `Wn=0.4`, which is used for all filter designs. A comparison between the amplitude responses shows the problem of compromising the highest first-sidelobe attenuation possible and the smallest possible pass-band. Considering the trend of the Kaiser vs. Blackman window: the free parameter $\beta$ is advantageous. At almost similar steepness of the flank, the Kaiser window has a smaller pass-band ($\Omega_S/\pi \approx 0.5$ vs. $\Omega_S/\pi \approx 0.55$) and at the same time a better first-sidelobe ($a_N \approx 83,\text{dB}$ vs. $a_N \approx 75,\text{dB}$).

---

### b)

The portrayed trend shows the amplitude response of a **high-pass** filter with order $m=33$ and a normalized frequency limit of $0.4$. A Hamming window is used here.

<img src="./pngs/ldsv5a02_b1.png" width="250">

When designing a high-pass with Python’s function `scipy.signal.firwin()`, Python generates the following warning:

```
For highpass and bandstop filters, order must be even.
Order is being increased by 1.
```

The `firwin()` function designs filters of type I or II depending on whether the order is even or odd. When the order is odd (type II), the frequency response must have a zero at the Nyquist frequency $\Omega/\pi=1$, which contradicts the demanded high-pass behavior. Therefore, Python automatically increases the filter order (type I).

---

### c)

If the **group delay** is calculated using Python (e.g., `scipy.signal.group_delay()`), we get the following plot versus normalized frequency for the designed low-pass in exercise a) with $m=50$ (window function: Hamming):

<img src="./pngs/ldsv5a02_c1.png" width="250">

The plot shows a constant group delay and clarifies the required linearity of the system.